# Inferential Statistics — AI Engineering Interview Prep

Covers: hypothesis testing, t-tests, chi-square, ANOVA, A/B testing, p-values.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8")
print("Ready")

## 1. One-Sample t-test
Test if a sample mean differs from a known value.

In [ ]:
# Is average page load time > 3 seconds?
load_times = np.random.normal(loc=3.2, scale=0.5, size=30)
hypothesized_mean = 3.0

t_stat, p_value = stats.ttest_1samp(load_times, popmean=hypothesized_mean, alternative="greater")

print(f"Sample mean:    {load_times.mean():.3f}s")
print(f"t-statistic:    {t_stat:.3f}")
print(f"p-value (one-tailed): {p_value:.4f}")
print(f"Reject H₀ at α=0.05? {p_value < 0.05}")

# Confidence interval
ci = stats.t.interval(0.95, df=len(load_times)-1,
                       loc=load_times.mean(),
                       scale=stats.sem(load_times))
print(f"95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")

## 2. Two-Sample t-test
Compare means of two independent groups.

In [ ]:
control   = np.random.normal(50, 10, 100)
treatment = np.random.normal(54, 10, 100)  # 4-unit effect

# Levene test for equal variances first
lev_stat, lev_p = stats.levene(control, treatment)
print(f"Levene test p={lev_p:.3f} → equal variances: {lev_p > 0.05}")

t_stat, p_val = stats.ttest_ind(control, treatment, equal_var=(lev_p > 0.05))
print(f"\nTwo-sample t-test:")
print(f"  Control mean:   {control.mean():.2f}")
print(f"  Treatment mean: {treatment.mean():.2f}")
print(f"  t-statistic:    {t_stat:.3f}")
print(f"  p-value:        {p_val:.4f}")
print(f"  Reject H₀?     {p_val < 0.05}")

## 3. Chi-Square Test of Independence

In [ ]:
# Is ad placement related to click-through?
# Contingency table: rows=placement, cols=clicked/not_clicked
observed = np.array([
    [45, 155],   # Top placement
    [30, 170],   # Middle
    [20, 180],   # Bottom
])

chi2, p_val, dof, expected = stats.chi2_contingency(observed)

print("Observed:")
print(observed)
print(f"\nChi2={chi2:.3f}, p={p_val:.4f}, dof={dof}")
print(f"Reject independence? {p_val < 0.05}")
print(f"\nExpected (if independent):")
print(expected.round(1))

## 4. One-Way ANOVA

In [ ]:
# Compare model accuracy across 3 algorithms
logistic   = np.random.normal(0.82, 0.03, 20)
random_forest = np.random.normal(0.87, 0.02, 20)
svm        = np.random.normal(0.84, 0.025, 20)

f_stat, p_val = stats.f_oneway(logistic, random_forest, svm)
print(f"ANOVA: F={f_stat:.3f}, p={p_val:.4f}")
print(f"At least one group mean differs? {p_val < 0.05}")

# Post-hoc: pairwise comparisons (Bonferroni correction)
groups = {"Logistic": logistic, "RF": random_forest, "SVM": svm}
n_comparisons = 3
alpha_corrected = 0.05 / n_comparisons  # Bonferroni
print(f"\nPost-hoc pairwise t-tests (Bonferroni α={alpha_corrected:.4f}):")
pairs = [("Logistic", "RF"), ("Logistic", "SVM"), ("RF", "SVM")]
for g1, g2 in pairs:
    t, p = stats.ttest_ind(groups[g1], groups[g2])
    print(f"  {g1} vs {g2}: p={p:.4f}  significant={p < alpha_corrected}")

## 5. A/B Testing Simulation

In [ ]:
# A/B test: does new checkout button increase conversion?
np.random.seed(0)
n_per_group = 1000

# Simulate: control 10% conversion, treatment 12%
control_conversions   = np.random.binomial(1, 0.10, n_per_group)
treatment_conversions = np.random.binomial(1, 0.12, n_per_group)

p_ctrl = control_conversions.mean()
p_trt  = treatment_conversions.mean()
lift   = (p_trt - p_ctrl) / p_ctrl * 100

# Proportions z-test
from statsmodels.stats.proportion import proportions_ztest
count = np.array([treatment_conversions.sum(), control_conversions.sum()])
nobs  = np.array([n_per_group, n_per_group])
z_stat, p_val = proportions_ztest(count, nobs)

print(f"Control rate:   {p_ctrl:.2%}")
print(f"Treatment rate: {p_trt:.2%}")
print(f"Relative lift:  {lift:.1f}%")
print(f"z-statistic:    {z_stat:.3f}")
print(f"p-value:        {p_val:.4f}")
print(f"Statistically significant? {p_val < 0.05}")

## 6. Type I/II Errors & Statistical Power

In [ ]:
from scipy.stats import norm

mu_null = 0
mu_alt  = 1.5    # true effect size
sigma   = 2.0
n       = 50
alpha   = 0.05

se = sigma / np.sqrt(n)
z_critical = norm.ppf(1 - alpha)  # one-tailed
critical_value = mu_null + z_critical * se

power = 1 - norm.cdf(critical_value, loc=mu_alt, scale=se)
beta  = 1 - power   # Type II error

print(f"Effect size (Cohen's d): {(mu_alt-mu_null)/sigma:.3f}")
print(f"Critical value:          {critical_value:.3f}")
print(f"Type I error (α):        {alpha}")
print(f"Type II error (β):       {beta:.3f}")
print(f"Statistical Power:       {power:.3f}")
print(f"\nPower > 0.80? {power > 0.80}")

## Interview Tips

- **p-value ≠ P(H₀ is true)**. It's P(data this extreme | H₀ true). Common misconception!
- **α=0.05 is convention**, not truth. Business context matters (medical vs. web test).
- **Multiple testing**: running 20 tests at α=0.05 expects ~1 false positive. Use Bonferroni or FDR.
- **Practical vs. statistical significance**: p<0.05 with n=1M means the effect might be trivially small.
- **Power of 0.80** is standard convention; higher power requires more samples.
- **Paired t-test** when same subjects measured twice (removes between-subject variance).